# 06a_Unsupervised_Model_Development_K-Means

AI Assistance:
OpenAI ChatGPT and Anthropic's Claude were used for code debugging, code generation, code organization,
and code methodological brainstorming. All final modeling, implementation,
validation, commentary, and interpretation were performed and verified by the authors.

In [ ]:
# pip install --upgrade ipywidgets jupyter

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
import umap
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)
import joblib
from sklearn.manifold import TSNE

In [ ]:
# Load the parquet file
INPUT_PATH = '../data/processed/processed_narratives.parquet'
df_narratives = pd.read_parquet(INPUT_PATH)

In [ ]:
df_narratives.head()

In [ ]:
# Drop duplicate narratives
starting_records = len(df_narratives)

df_narratives = (
    df_narratives
    .drop_duplicates(subset=['processed_narrative'])
    .reset_index(drop=True)
)

ending_records = len(df_narratives)

print(f'Records before dropping duplicates: {starting_records:,}')
print(f'Duplicate narratives removed: {starting_records - ending_records:,}')
print(f'Records after dropping duplicates: {ending_records:,}')

## Part 1 - K-Means, No Stopwords

In [ ]:
# # If re-running notebook, load the trained Vectorizer
# tfidf = joblib.load('../data/processed/tfidf_vectorizer.joblib')
#
# # Transform rather than re-fit
# X_tfidf = tfidf.transform(
#     df_narratives['processed_narrative']
# )
#
# print(X_tfidf.shape)

In [ ]:
# Get TF-IDF weights
tfidf = TfidfVectorizer(
    min_df=10,
    max_df=0.95,
    max_features=20000,
    ngram_range=(1, 2),
)

X_tfidf = tfidf.fit_transform(df_narratives['processed_narrative'])

print(X_tfidf.shape)

In [ ]:
# Get top TF-IDF terms
feature_names = np.array(tfidf.get_feature_names_out())

mean_tfidf = np.asarray(X_tfidf.mean(axis=0)).ravel()

top_tfidf_terms = (
    pd.DataFrame({
        'term': feature_names,
        'mean_tfidf': mean_tfidf
    })
    .sort_values('mean_tfidf', ascending=False)
)

top_tfidf_terms.head(25)

In [ ]:
# Generate the elbow plot

# In the scikit-learn documentation, the inertia attribute is defined
# as the sum of squared distances of samples to their closest cluster
# center.

k_values = [2, 4, 6, 8, 10, 12, 14, 16, 18, 20]
wgss = []

for k in k_values:
    kmeans = MiniBatchKMeans(
        n_clusters=k,
        init='k-means++',
        batch_size=1000,
        random_state=42,
        n_init=1,
        max_iter=100
    )

    kmeans.fit(X_tfidf)

    # inertia_ = within-cluster sum of squares / WGSS
    wgss.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_values, wgss, marker='o')
plt.xlabel('Number of clusters')
plt.ylabel('Within groups sum of squares')
plt.title('Elbow Plot: MiniBatchKMeans')
plt.xticks(k_values)
plt.grid(True)
plt.show()

In [ ]:
# Try different parameters for K-Means
n_clusters_grid = [10, 12, 14, 16, 18, 20]

sample_idx = np.random.RandomState(42).choice(
    X_tfidf.shape[0],
    size=10000,
    replace=False
)

X_sample_sparse = X_tfidf[sample_idx]
X_sample_dense = X_sample_sparse.toarray()

results = []

for k in n_clusters_grid:
    model = MiniBatchKMeans(
        n_clusters=k,
        init='k-means++',
        batch_size=1000,
        random_state=42,
        n_init=1,
        max_iter=100
    )

    cluster_labels = model.fit_predict(X_tfidf)
    cluster_sample = cluster_labels[sample_idx]

    results.append({
        'n_clusters': k,
        'inertia': model.inertia_,
        'silhouette_cosine': silhouette_score(
            X_sample_sparse,
            cluster_sample,
            metric='cosine'
        ),
        'davies_bouldin': davies_bouldin_score(
            X_sample_dense,
            cluster_sample
        ),
        'calinski_harabasz': calinski_harabasz_score(
            X_sample_dense,
            cluster_sample
        )
    })

results_df = (
    pd.DataFrame(results)
    .sort_values(
        by=['silhouette_cosine', 'davies_bouldin', 'calinski_harabasz'],
        ascending=[False, True, False]
    )
)

results_df

In [ ]:
# # If re-running notebook, load the trained K-Means
# kmeans = joblib.load('../data/processed/kmeans_clusters.joblib')
#
# # Predict rather than re-fit
# df_narratives['cluster'] = kmeans.fit_predict(X_tfidf)
# df_narratives['cluster'].value_counts().sort_index()

In [ ]:
# Run K-Means with k = 18 based on elbow plot and parameter search
k = 18

kmeans = MiniBatchKMeans(
    n_clusters=k,
    init='k-means++',
    batch_size=1000,
    random_state=42,
    n_init=1,
    max_iter=100
)

df_narratives['cluster'] = kmeans.fit_predict(X_tfidf)
df_narratives['cluster'].value_counts().sort_index()

In [ ]:
# Get top terms for each cluster
terms = np.array(tfidf.get_feature_names_out())

for cluster_num in range(kmeans.n_clusters):
    top_indices = kmeans.cluster_centers_[cluster_num].argsort()[::-1][:8]
    top_terms = terms[top_indices]

    print(f'\nCluster {cluster_num}')
    print(', '.join(top_terms))

In [ ]:
# Add distance to assigned centroid to dataframe
distances = kmeans.transform(X_tfidf)
assigned_cluster = df_narratives['cluster'].values

df_narratives['distance_to_centroid'] = (
    distances[
        np.arange(len(df_narratives)),
        assigned_cluster
    ]
)

In [ ]:
# Save fitted model and vectorizer

joblib.dump(
    tfidf,
    '../data/processed/tfidf_vectorizer.joblib'
)

joblib.dump(
    kmeans,
    '../data/processed/kmeans_clusters.joblib'
)

print('Model saved.')

### UMAP and t-SNE Visualizations

In [ ]:
# Take sample to use for UMAP
sample_size = 20000

sample_idx = df_narratives.sample(
    n=sample_size,
    random_state=42
).index

X_sample = X_tfidf[sample_idx]

cluster_sample = (
    df_narratives
    .loc[sample_idx, 'cluster']
    .astype(int)
)

In [ ]:
# Run SVD on 20k sample
svd = TruncatedSVD(
    n_components=50,
    random_state=42
)

X_sample_svd = svd.fit_transform(X_sample)

In [ ]:
# Run UMAP
umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)

X_umap = umap_model.fit_transform(X_sample_svd)

In [ ]:
# Plot the UMAP
k = 18

cmap = plt.colormaps['tab20'].resampled(k)

plt.figure(figsize=(10, 8))

scatter = plt.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    c=cluster_sample,
    cmap=cmap,
    s=5,
    alpha=0.6
)

plt.title('UMAP Projection of Complaint Clusters')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')

cbar = plt.colorbar(
    scatter,
    ticks=range(0, k, 5)
)

cbar.set_label('Cluster')

plt.show()

In [ ]:
# Initialize t-SNE
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate='auto',
    init='pca',
    random_state=42
)

X_tsne = tsne.fit_transform(X_sample_svd)

In [ ]:
# Plot t-SNE
plt.figure(figsize=(10, 8))

scatter = plt.scatter(
    X_tsne[:, 0],
    X_tsne[:, 1],
    c=cluster_sample,
    cmap=plt.colormaps['tab20'].resampled(18),
    s=5,
    alpha=0.6
)

plt.colorbar(scatter)
plt.title('t-SNE Projection of Complaint Clusters')
plt.show()

In [ ]:
df_narratives.head()

In [ ]:
df_narratives.info()

In [ ]:
df_narratives['cluster'].value_counts().sort_index(ascending=False)

In [ ]:
# Export the final features
OUTPUT_PATH = '../data/processed/clustered_narratives.parquet'

df_narratives.to_parquet(
    OUTPUT_PATH,
    index=False
)

# Part 2 - K-Means, Company Stopwords

Note: This section was run purely for metric and visual comparison and the features were not exported for the rest of the workflow. Models were saved purely for computational convenience.

In [ ]:
domain_stopwords = {
    'goldman',
    'sachs',
    'one',
    'chase',
    'well',
    'fargo',
    'capital',
    'america',
    'citibank',
    'citi',
    'jpmorgan',
    'navy',
    'federal',
    'union',
    'american',
    'express',
    'amex',
}

custom_stopwords = ENGLISH_STOP_WORDS.union(domain_stopwords)

In [ ]:
# # If re-running notebook, load the trained Vectorizer
# tfidf = joblib.load('../data/processed/stopwords_tfidf_vectorizer.joblib')
#
# # Transform rather than re-fit
# X_tfidf = tfidf.transform(
#     df_narratives['processed_narrative']
# )
#
# print(X_tfidf.shape)

In [ ]:
# Get TF-IDF weights
tfidf = TfidfVectorizer(
    stop_words=list(custom_stopwords),
    min_df=10,
    max_df=0.95,
    max_features=20000,
    ngram_range=(1, 2),
)

X_tfidf = tfidf.fit_transform(df_narratives['processed_narrative'])

print(X_tfidf.shape)

In [ ]:
# Get top TF-IDF terms
feature_names = np.array(tfidf.get_feature_names_out())

mean_tfidf = np.asarray(X_tfidf.mean(axis=0)).ravel()

top_tfidf_terms = (
    pd.DataFrame({
        'term': feature_names,
        'mean_tfidf': mean_tfidf
    })
    .sort_values('mean_tfidf', ascending=False)
)

top_tfidf_terms.head(25)

In [ ]:
# # If re-running notebook, load the trained K-Means
# kmeans = joblib.load('../data/processed/stopwords_kmeans_clusters.joblib')
#
# # Predict rather than re-fit
# df_narratives['cluster'] = kmeans.fit_predict(X_tfidf)
# df_narratives['cluster'].value_counts().sort_index()

In [ ]:
# Run K-Means with k = 18 for direct comparison with Part 1
k = 18

kmeans = MiniBatchKMeans(
    n_clusters=k,
    init='k-means++',
    batch_size=1000,
    random_state=42,
    n_init=1,
    max_iter=100
)

df_narratives['cluster'] = kmeans.fit_predict(X_tfidf)
df_narratives['cluster'].value_counts().sort_index()

In [ ]:
def calculate_kmeans_silhouette(
    model,
    X_tfidf,
    sample_size=10000,
    random_state=42,
    metric='cosine',
    cluster_labels=None
):
    """
    Calculate silhouette score for a trained or pre-trained K-Means model.

    If cluster_labels are provided, the function uses them directly.
    Otherwise, it uses model.predict(X_tfidf), which works for a loaded model.
    """

    sample_idx = np.random.RandomState(random_state).choice(
        X_tfidf.shape[0],
        size=min(sample_size, X_tfidf.shape[0]),
        replace=False
    )

    X_sample_sparse = X_tfidf[sample_idx]

    if cluster_labels is None:
        cluster_labels = model.predict(X_tfidf)

    cluster_sample = cluster_labels[sample_idx]

    sil_score = silhouette_score(
        X_sample_sparse,
        cluster_sample,
        metric=metric
    )

    print(f'Silhouette score ({metric}): {sil_score:.4f}')

    return sil_score

In [ ]:
# # Get silhouette score for pre-trained model
# kmeans = joblib.load('kmeans_model.pkl')
#
# sil_score = calculate_kmeans_silhouette(
#     model=kmeans,
#     X_tfidf=X_tfidf
# )

In [ ]:
# Get the silhouette score
cluster_labels = kmeans.fit_predict(X_tfidf)

sil_score = calculate_kmeans_silhouette(
    model=kmeans,
    X_tfidf=X_tfidf,
    cluster_labels=cluster_labels
)

In [ ]:
# Get top terms for each cluster
terms = np.array(tfidf.get_feature_names_out())

cluster_terms = []

for cluster_num in range(k):
    top_indices = kmeans.cluster_centers_[cluster_num].argsort()[::-1][:8]
    top_terms = terms[top_indices]
    cluster_terms.append({
        'cluster': cluster_num,
        'top_terms': ', '.join(top_terms)
    })

    print(f'\nCluster {cluster_num}')
    print(', '.join(top_terms))

pd.DataFrame(cluster_terms).to_csv(
    '../data/processed/cluster_top_terms.csv',
    index=False
)

print('\nCluster top terms saved.')

In [ ]:
# Save fitted model and vectorizer
joblib.dump(
    tfidf,
    '../data/processed/stopwords_tfidf_vectorizer.joblib'
)

joblib.dump(
    kmeans,
    '../data/processed/stopwords_kmeans_clusters.joblib'
)

print('Model saved.')

### UMAP

In [ ]:
# Take sample to use for UMAP
sample_size = 20000

sample_idx = df_narratives.sample(
    n=sample_size,
    random_state=42
).index

X_sample = X_tfidf[sample_idx]

cluster_sample = (
    df_narratives
    .loc[sample_idx, 'cluster']
    .astype(int)
)

In [ ]:
# Run SVD on 20k sample
svd = TruncatedSVD(
    n_components=50,
    random_state=42
)

X_sample_svd = svd.fit_transform(X_sample)

In [ ]:
# Run UMAP
umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)

X_umap = umap_model.fit_transform(X_sample_svd)

In [ ]:
# Plot the UMAP
k = 18

cmap = plt.colormaps['tab20'].resampled(k)

fig, ax = plt.subplots(figsize=(10, 8))

scatter = ax.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    c=cluster_sample,
    cmap=cmap,
    s=5,
    alpha=0.6
)

# Annotate each cluster at its median UMAP position
cluster_arr = cluster_sample.values
for c in range(k):
    mask = cluster_arr == c
    if mask.sum() == 0:
        continue
    cx = np.median(X_umap[mask, 0])
    cy = np.median(X_umap[mask, 1])
    ax.text(
        cx, cy, str(c),
        fontsize=9, fontweight='bold',
        ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.6, ec='none')
    )

ax.set_title('UMAP Projection of Complaint Clusters')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')

cbar = plt.colorbar(scatter, ax=ax, ticks=range(0, k, 5))
cbar.set_label('Cluster')

plt.show()